# Huni - ground-truth extractor

**Upload an original MP3, get back the `{syllable, start, end, note}` JSON the app loads.**

Demucs isolates the vocal -> torchcrepe tracks pitch -> the contour is
segmented into notes -> Whisper supplies word timings from the same stem.

### Before you start
1. **Runtime > Change runtime type > T4 GPU.**
2. Upload the **original recording**, not a karaoke/minus-one version - the
   method extracts a vocal, and a minus-one has none.

Deliberately no TensorFlow: it claims the whole GPU on import, leaving
nothing for Whisper. torchcrepe is the same CREPE algorithm in PyTorch, so
pitch tracking and transcription share one allocator.


In [ ]:
#@title 1 - Install
!pip install -q demucs openai-whisper torchcrepe soundfile librosa

import os, re, json, glob, gc, subprocess
import numpy as np, librosa, torch, torchcrepe, whisper

print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0),
          f'{torch.cuda.mem_get_info()[0]/1e9:.1f} GB free')
else:
    print('NO GPU - Runtime > Change runtime type > T4 GPU')


In [ ]:
#@title 2 - Upload original MP3(s)
import os, re
from google.colab import files

print('Select ORIGINAL mp3 files (not karaoke/minus-one versions)...')
uploaded = files.upload()
paths = list(uploaded)

def clean_title(filename):
    """Download names carry artist and video tags, but the JSON filename has
    to match what the catalog calls the song or the app will not load it.
    'Adie - Tahanan (Official Lyric Video) (2).mp3' -> 'Tahanan'"""
    t = os.path.splitext(filename)[0]
    t = re.sub(r'\((?:official|lyric|music|video|audio|hd|mv)[^)]*\)', '', t, flags=re.I)
    t = re.sub(r'\[[^\]]*\]', '', t)
    t = re.sub(r'\(\s*\d+\s*\)', '', t)
    if ' - ' in t:
        t = t.split(' - ')[-1]
    return re.sub(r'\s+', ' ', t).strip()

TITLES = {p: clean_title(p) for p in paths}
for p in paths:
    print(f'  {os.path.getsize(p)/1e6:5.1f} MB  {p}')
    print(f'              -> "{TITLES[p]}"')
print()
print('Fix any wrong title before Cell 4, e.g.:')
print('   TITLES[paths[0]] = "Araw-Araw"')


In [ ]:
#@title 3 - Settings
LANGUAGE        = 'tl'        #@param ['tl','en','auto']
WHISPER_MODEL   = 'large-v3'  #@param ['large-v3','medium','small']
CONF_MIN        = 0.50        #@param {type:"number"}
SEMITONE_CHANGE = 0.8         #@param {type:"number"}
MIN_NOTE_MS     = 100         #@param {type:"integer"}

# CONF_MIN         below this the tracker is not on a real pitch
# SEMITONE_CHANGE  pitch movement that starts a new note
# MIN_NOTE_MS      shorter than this is a blip, not a note
print('ok')


In [ ]:
#@title 4 - Extract ground truth
import os, re, json, glob, gc, subprocess
import numpy as np, librosa, torch, torchcrepe, whisper

NAMES = ['C', 'C♯', 'D', 'D♯', 'E', 'F',
         'F♯', 'G', 'G♯', 'A', 'A♯', 'B']
_asr = None

def free_gpu():
    """Demucs, the pitch tracker and whisper each want GPU memory. Released
    between stages they fit on a T4; held at once they do not."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def load_asr():
    """large-v3 transcribes best but wants ~10 GB. If it will not fit,
    medium is still far better than the tiny model the phone runs."""
    global _asr, WHISPER_MODEL
    if _asr is None:
        free_gpu()
        try:
            print(f'loading whisper {WHISPER_MODEL} (first time downloads it)...')
            _asr = whisper.load_model(WHISPER_MODEL)
        except torch.cuda.OutOfMemoryError:
            print(f'  {WHISPER_MODEL} did not fit -> falling back to medium')
            free_gpu()
            WHISPER_MODEL = 'medium'
            _asr = whisper.load_model('medium')
    return _asr

def isolate_vocal(mp3):
    """Demucs writes separated/<model>/<stem>/vocals.wav; the model folder
    name changes between versions, so the file is found by glob."""
    subprocess.run(['demucs', '--two-stems=vocals', '-o', '/content/sep', mp3],
                   check=True)
    stem = os.path.splitext(os.path.basename(mp3))[0]
    hits = glob.glob(f'/content/sep/*/{glob.escape(stem)}/vocals.wav')
    if not hits:
        raise FileNotFoundError('Demucs produced no vocal stem')
    free_gpu()
    return hits[0]

def track_pitch(vocal_wav):
    """-> (times, f0_hz, confidence) on a 10 ms grid."""
    audio, sr = librosa.load(vocal_wav, sr=16000, mono=True)
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    hop = int(sr / 100)
    x = torch.tensor(audio, dtype=torch.float32, device=dev)[None]
    f0, per = torchcrepe.predict(
        x, sr, hop, fmin=50.0, fmax=1100.0, model='full',
        decoder=torchcrepe.decode.viterbi, batch_size=512,
        device=dev, return_periodicity=True)
    f0, conf = f0.squeeze().cpu().numpy(), per.squeeze().cpu().numpy()
    del x, per
    free_gpu()
    return np.arange(len(f0)) * (hop / sr), f0, conf

def segment_notes(t, f0, conf):
    """Continuous pitch -> discrete notes. A note ends when the pitch moves
    more than SEMITONE_CHANGE from its running median, or the voice stops."""
    midi = np.where(f0 > 0, 69 + 12 * np.log2(np.maximum(f0, 1e-6) / 440.0), np.nan)
    voiced = conf >= CONF_MIN

    out, start, buf = [], None, []
    for i in range(len(t)):
        if voiced[i] and np.isfinite(midi[i]):
            if start is None:
                start, buf = t[i], [midi[i]]
            elif abs(midi[i] - np.median(buf)) > SEMITONE_CHANGE:
                out.append((start, t[i], float(np.median(buf))))
                start, buf = t[i], [midi[i]]
            else:
                buf.append(midi[i])
        elif start is not None:
            out.append((start, t[i], float(np.median(buf))))
            start, buf = None, []
    if start is not None:
        out.append((start, t[-1], float(np.median(buf))))
    return [n for n in out if (n[1] - n[0]) * 1000 >= MIN_NOTE_MS]

def get_words(vocal_wav):
    """Word timings from the ISOLATED vocal - the backing track is exactly
    what makes transcription hard."""
    kw = {'word_timestamps': True}
    if LANGUAGE != 'auto':
        kw['language'] = LANGUAGE
    res = load_asr().transcribe(vocal_wav, **kw)
    free_gpu()
    return [{'w': w['word'].strip(), 's': w['start'], 'e': w['end']}
            for seg in res['segments'] for w in seg.get('words', [])
            if w['word'].strip()]

def build(mp3):
    print(f'\n=== {mp3} ===')
    vocal = isolate_vocal(mp3)

    t, f0, conf = track_pitch(vocal)
    notes = segment_notes(t, f0, conf)
    print(f'  {len(notes)} notes')

    words = get_words(vocal)
    print(f'  {len(words)} words')

    def word_at(s, e):
        best, best_ov = '', 0.0
        for w in words:
            ov = min(e, w['e']) - max(s, w['s'])
            if ov > best_ov:
                best_ov, best = ov, w['w']
        return best

    data = [{'syllable': word_at(s, e),
             'start': round(float(s), 3),
             'end': round(float(e), 3),
             'note': f'{NAMES[int(round(m)) % 12]}{int(round(m)) // 12 - 1}'}
            for s, e, m in notes]

    title = TITLES.get(mp3, os.path.splitext(os.path.basename(mp3))[0])
    safe = re.sub(r'[^A-Za-z0-9]+', '_', title).strip('_')
    out = f'/content/{safe}.json'
    json.dump({'data': data}, open(out, 'w', encoding='utf-8'),
              ensure_ascii=False, indent=1)
    print(f'  -> {out}  ({len(data)} entries, title "{title}")')
    free_gpu()
    return out, data

results = {}
for p in paths:
    try:
        results[p] = build(p)
    except Exception as exc:
        print(f'  FAILED: {type(exc).__name__}: {exc}')
        free_gpu()

print(f'\ndone: {len(results)}/{len(paths)} songs')


In [ ]:
#@title 5 - Preview
for src, (path, data) in results.items():
    print(f'\n=== {src} - {len(data)} entries ===')
    for d in data[:12]:
        print(f"  {d['start']:7.3f}-{d['end']:7.3f}  {d['note']:<4}  {d['syllable']}")
    named = sum(1 for d in data if d['syllable'])
    print(f"  with a word attached : {named}/{len(data)} "
          f"({100*named/max(len(data),1):.0f}%)")
    if data:
        print(f"  covers               : {data[-1]['end']-data[0]['start']:.1f}s")


## 6 - Validate before trusting it

Generated ground truth decides what the app tells singers they got wrong.
Run this notebook on **Nasa Iyo Na Ang Lahat**, then upload the existing
`assets/ground_truth/Nasa_Iyo_Na_Ang_Lahat.json` and compare.


In [ ]:
#@title 6 - Agreement with the hand-authored file, then download
import re, json
from google.colab import files

print('Upload the hand-authored JSON to compare against (Cancel to skip)')
try:
    ref_up = files.upload()
    ref = json.load(open(list(ref_up)[0], encoding='utf-8'))['data']
    gen = list(results.values())[0][1]

    def at(entries, when):
        for e in entries:
            if e['start'] <= when <= e['end']:
                return e
        return None

    exact = chroma = covered = 0
    for r in ref:
        g = at(gen, (r['start'] + r['end']) / 2)
        if not g:
            continue
        covered += 1
        exact += g['note'] == r['note']
        # Same pitch class, wrong octave: the classic tracker error, worth
        # separating since it has a different cause than being plain wrong.
        chroma += re.sub(r'\d', '', g['note']) == re.sub(r'\d', '', r['note'])

    print()
    print(f'reference entries    : {len(ref)}')
    print(f'covered by generated : {covered} ({100*covered/len(ref):.0f}%)')
    print(f'exact note match     : {exact}/{covered} = {100*exact/max(covered,1):.1f}%')
    print(f'same pitch class     : {chroma}/{covered} = {100*chroma/max(covered,1):.1f}%')
    print()
    print('>=80% exact : trustworthy enough to generate the rest')
    print('50-80%      : usable only if you say so in the writeup')
    print('<50%        : singers scored against wrong notes - do not ship')
except Exception as e:
    print('skipped validation:', e)

for path, _ in results.values():
    files.download(path)
print()
print('Put these in assets/ground_truth/ - filename must match the song title.')
